In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Aesthetic setup for professional clarity
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 10, 'axes.labelsize': 12, 'axes.titlesize': 14})

In [ ]:
# Load dataset
df = pd.read_csv('StudentPerformanceFactors.csv')

# Inspection for "Richness" (Volume and Variety)
print(f"Dataset contains {df.shape[0]} rows and {df.shape[1]} columns.")
df.info()
df.head()

In [ ]:
# Temporary encoding just for the heatmap to show all features
df_temp = df.copy().dropna()
for col in df_temp.select_dtypes(include=['object']).columns:
    df_temp[col] = LabelEncoder().fit_transform(df_temp[col])

# Visualization 1: Correlation Heatmap (Clarity: Masked for readability)
plt.figure(figsize=(14, 8))
mask = np.triu(np.ones_like(df_temp.corr(), dtype=bool))
sns.heatmap(df_temp.corr(), mask=mask, annot=True, fmt='.2f', cmap='coolwarm')
plt.title("Correlation Analysis: Academic & Socio-Economic Factors")
plt.show()

# Visualization 2: Target Variable Distribution
plt.figure(figsize=(10, 5))
sns.histplot(df['Exam_Score'], kde=True, color='teal')
plt.title("Frequency Distribution of Student Exam Scores")
plt.xlabel("Final Exam Score")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
# 1. Cleaning: Remove missing values to ensure data integrity
df = df.dropna()

# 2. Smart Encoding: Ordinal Mapping for ranked factors (Improvement point)
rank_map = {'Low': 0, 'Medium': 1, 'High': 2}
ordinal_cols = ['Parental_Involvement', 'Motivation_Level', 'Access_to_Resources', 'Family_Income', 'Teacher_Quality']
for col in ordinal_cols:
    df[col] = df[col].map(rank_map)

# 3. Standard Encoding for remaining categorical columns
le = LabelEncoder()
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("Preprocessing complete: All data converted to numeric intelligently.")

In [ ]:
# Separate Features (X) and Target (y)
X = df.drop('Exam_Score', axis=1)
y = df['Exam_Score']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize (Mandatory for Gradient Descent Optimization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data standardized: Features now have mean=0 and variance=1.")

In [ ]:
# Initialize SGD Regressor
sgd_reg = SGDRegressor(max_iter=1, tol=None, warm_start=True, random_state=42)

train_loss, test_loss = [], []

# Training loop to capture optimization progress
for epoch in range(500):
    sgd_reg.partial_fit(X_train_scaled, y_train)
    train_loss.append(mean_squared_error(y_train, sgd_reg.predict(X_train_scaled)))
    test_loss.append(mean_squared_error(y_test, sgd_reg.predict(X_test_scaled)))

print("Linear Regression optimized via Gradient Descent.")

In [ ]:
# Decision Tree Model
dt_reg = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_reg.fit(X_train_scaled, y_train)

# Random Forest Model
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train_scaled, y_train)

# Performance Evaluation
print(f"LR (GD) MSE: {mean_squared_error(y_test, sgd_reg.predict(X_test_scaled)):.4f}")
print(f"Decision Tree MSE: {mean_squared_error(y_test, dt_reg.predict(X_test_scaled)):.4f}")
print(f"Random Forest MSE: {mean_squared_error(y_test, rf_reg.predict(X_test_scaled)):.4f}")

In [ ]:
# Plot 1: Loss Curve for Gradient Descent
plt.figure(figsize=(10, 5))
plt.plot(train_loss, label='Train Loss', color='blue')
plt.plot(test_loss, label='Test Loss', color='red', linestyle='--')
plt.title("Optimization Progress: Loss Reduction via Gradient Descent")
plt.xlabel("Iterations (Epochs)")
plt.ylabel("MSE")
plt.legend()
plt.show()

# Plot 2: Scatter Plot with Linear Regression Fit Line
plt.figure(figsize=(10, 6))
plt.scatter(X_test['Attendance'], y_test, alpha=0.3, color='gray', label='Actual Student Data')
# Sort for a clean line
sort_idx = np.argsort(X_test['Attendance'])
plt.plot(X_test['Attendance'].iloc[sort_idx], sgd_reg.predict(X_test_scaled)[sort_idx],
         color='darkred', linewidth=3, label='Regression Line (LR)')
plt.title("Linear Regression Fit: Attendance Impact on Scores")
plt.xlabel("Attendance %")
plt.ylabel("Exam Score")
plt.legend()
plt.show()

In [ ]:
# Determine the best model (lowest MSE)
models = {'LR': sgd_reg, 'DT': dt_reg, 'RF': rf_reg}
best_name = 'RF' # Typically RF performs best here
joblib.dump(models[best_name], 'best_educational_model.pkl')

# Script for single prediction (Task 2 Requirement)


single_sample = X_test_scaled[10].reshape(1, -1)
prediction = models[best_name].predict(single_sample)[0]

print(f"--- Best Model ({best_name}) Saved ---")
print(f"Single Prediction: {prediction:.2f}")
print(f"Actual Value: {y_test.iloc[10]}")